In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub
import os

import warnings
warnings.filterwarnings('ignore')

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
Q1_data_path = os.path.join(path, 'Q1_data.csv') #Saved the path in a variable for clean code
df_Q1_data = pd.read_csv(Q1_data_path) # Reading the data using read_csv and saving it to df_Q1_data

In [ ]:
# Task 2: Write your code here:
df_Q1_data.head() # inspecting first five rows using head()


In [ ]:
# Task 3: Write your code here:
df_Q1_data.info() # displaying dataset information using info()

# Order_ID, Distance_km, Vehicle_Type, Preparation_Time_min has 0 Null count
# There are 4 categorical columns that might need some encoding

In [ ]:
# Task 4: Write your code here:

df_Q1_data.describe()  # Showing statistical description using describe()

# We can see that Delivery_Time has outliers and we might need some scaling

In [ ]:
# Task 5: Write your code here:


# Ploting the target distribution (delivery_time)

plt.figure(figsize=(10, 5))
plt.hist(df_Q1_data['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

# after ploting we can see clearly the outliers

In [ ]:
# Task 1: Write your code here:
df_Q1_data = df_Q1_data.drop(columns=['Order_ID'])

df_Q1_data

In [ ]:
# Task 2: Write your code here:

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_Q1_data)

missing_ccolumns = ['Weather', 'Traffic_Level', 'Time_of_Day']
missing_ncolumns = ['Courier_Experience_yrs', 'Delivery_Time']

for col in missing_ccolumns:
    df_Q1_data[col] = df_Q1_data[col].fillna('unknown')
for col in missing_ccolumns:
    df_Q1_data[col] = df_Q1_data[col].fillna(df_Q1_data[col].mode()[0])

for col in missing_ncolumns:
    df_Q1_data[col] = df_Q1_data[col].fillna(df_Q1_data[col].mean())

check_missing_values(df_Q1_data)


In [ ]:
# Task 3: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_Q1_data)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder #import OneHotEncoder


label_encoder = LabelEncoder()
 # Instantiate OneHotEncoder
#df_Q1_data = onehot_encoder.fit_transform(df_Q1_data) # Apply fit_transform to the copied
target_column = 'Delivery_Time'

X = df_Q1_data.drop(columns=[target_column])
y = df_Q1_data[target_column]

for col in X.select_dtypes(include=["object"]).columns:
    X[col] = label_encoder.fit_transform(X[col])

X

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
# Task 6: Write your code here:



In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae_scores = []
rmse_scores = []

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)
y_pred = model.predict(X_test_scaled)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:

feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
       'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=30, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# Task Bonus: Write your code here: